# Baseline: default Spraypoint

What does the network look like with no bypass? This sets the numbers the bypass has to beat.

Reads the baseline runs of the sweep set by `SWEEP` in `common.py`, so it shows whatever runs have finished. Rerun it as more arrive.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import common

common.setup_plots()
pd.options.display.float_format = "{:,.2f}".format

runs = common.load_runs(common.BASELINE_EXPERIMENT)
flows = common.load_flows(runs)
links = common.load_links(runs)
print(f"Reading {common.MACHINE}/{common.BATCH}")
print(f"{len(runs)} finished runs, {', '.join(common.SETTING_LABELS[s].lower() for s in common.SETTINGS)} {sorted({r.setting for r in runs})}, seeds {sorted({r.seed for r in runs})}")

## Are the runs healthy?

`unfinished`, `ttl_drops` and `no_route_drops` should all be 0. Queue drops are normal under load.

In [ ]:
health = common.runs_table(runs)
display(health)

problems = health[(health["unfinished"] > 0) | (health["ttl_drops"] > 0) | (health["no_route_drops"] > 0)]
print("All runs healthy" if problems.empty else f"Check these runs:\n{problems}")

## What traffic is it?

Small flows are most of the flows but few of the bytes, so a bypass that moves only small flows moves little traffic. The adjacent class is the share of router pairs that are neighbors, about *d*/(*n* − 1).

In [ ]:
display(common.traffic_mix(flows, common.SETTINGS).unstack("class"))

## Flow completion times

Pooled across seeds, leaving out flows that start in the first and last 10% of a run, when the network sees less load. `beyond_p99` is how many flows the p99 rests on: below about 100, treat it as noisy, and the same goes for p99.9 at 10 times that flow count.

In [ ]:
display(common.fct_table(flows, ["class", *common.SETTINGS]))

### The tail of the distribution

Each line shows the share of flows slower than a given time, so the far right of each panel is the tail. A jump near 1,000 µs or 10,000 µs is usually a retransmission timeout or a lost SYN, not queueing.

In [ ]:
colors = common.setting_colors(flows[common.SETTING])
for title, rows in common.panels(flows):
    figure, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
    for axis, flow_class in zip(axes, common.CLASSES):
        for value, group in rows[rows["class"] == flow_class].groupby(common.SETTING):
            x, y = common.tail_curve(group["fct_us"].to_numpy())
            axis.step(x, y, color=colors[value], label=common.setting_name(value), where="post")
        axis.set(title=common.CLASS_NAMES[flow_class], xscale="log", yscale="log", xlabel="FCT (µs)")
    axes[0].set_ylabel("Share of flows slower")
    axes[0].legend()
    if title:
        figure.suptitle(title)
    figure.tight_layout()

### Where the tail comes from

A small flow that loses nothing finishes within a few hundred microseconds, so one that takes 1 ms or more waited for a timeout: the 1 ms minimum RTO for lost data, or the 10 ms handshake timeout for a lost SYN or SYN-ACK. A class's p99 jumps to about 1,000 µs once more than 1% of its flows wait for a timeout, and to about 10,000 µs once more than 1% wait for a handshake timeout. So the tail here counts losses more than it measures queueing.

`clean_p50`, `clean_p99` and `clean_max` describe the flows with no timeout, which is the queueing alone. The grey line in the plot is 1%, where the p99 jumps.

In [ ]:
display(common.timeout_table(flows, ["class", *common.SETTINGS]))

In [ ]:
timeouts = common.timeout_table(flows, ["class", *common.SETTINGS, "seed"]).reset_index()
for title, rows in common.panels(timeouts):
    figure, axes = plt.subplots(1, 2, figsize=(8, 3.4), sharey=True)
    for axis, flow_class in zip(axes, common.SMALL_CLASSES):
        data = rows[rows["class"] == flow_class].groupby(common.SETTING)[["data_timeout_pct", "handshake_timeout_pct"]].mean()
        axis.plot(data.index, data["data_timeout_pct"], color="#2a78d6", marker="o", label="Data (1 ms)")
        axis.plot(data.index, data["handshake_timeout_pct"], color="#104281", marker="s", linestyle="--", label="Handshake (10 ms)")
        axis.axhline(1, color="#b5b4ad", linewidth=1)
        axis.set(title=common.CLASS_NAMES[flow_class])
        common.label_setting_axis(axis)
    axes[0].set_ylim(bottom=0)
    axes[0].set_ylabel("Flows waiting for a timeout (%)")
    axes[0].legend(title="Timeout")
    if title:
        figure.suptitle(title)
    figure.tight_layout()

## Large-flow throughput

Each large flow's goodput is its bytes over its FCT, against 100 Gbps links. `p1` is the slowest 1% of flows, which a hotspot would hurt first.

In [ ]:
display(common.goodput_table(flows, common.SETTINGS))

## Paths

Share of delivered packets, including ACKs, at each number of fabric hops. Spraying means adjacent routers still mostly use long paths: this is the gap the bypass targets.

In [ ]:
paths = common.path_table(runs).groupby(["class", "hops"])["fraction"].mean().reset_index()
figure, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for axis, flow_class in zip(axes, common.CLASSES):
    data = paths[(paths["class"] == flow_class) & (paths["hops"] <= 6)]
    axis.bar(data["hops"], data["fraction"], color=common.MODE_COLORS["none"], width=0.7)
    axis.set(title=common.CLASS_NAMES[flow_class], xlabel="Fabric hops")
    axis.grid(axis="x", visible=False)
axes[0].set_ylabel("Share of packets")
figure.tight_layout()

## Links

How evenly does traffic spread? `hotspot_ratio` is the busiest link's utilization over the mean. This is the baseline for H2's question of whether the bypass creates hotspots.

In [ ]:
display(common.link_table(links, common.SETTINGS))

In [ ]:
for title, rows in common.panels(links):
    figure, axis = plt.subplots(figsize=(6, 3.4))
    for value, group in rows.groupby(common.SETTING):
        utilization = np.sort(group["utilization"].to_numpy())
        axis.plot(utilization, np.linspace(0, 1, len(utilization)), color=colors[value], label=common.setting_name(value))
    axis.set(xlabel="Link utilization", ylabel="Share of links at or below", xlim=(0, None), title=title)
    axis.legend()
    figure.tight_layout()

## Fan-in

Spraypoint should bring a destination's traffic in through many neighbors. `busiest_share` is the share of its traffic arriving from its busiest neighbor, and `neighbors` is how many deliver anything.

In [ ]:
display(common.fan_in_table(links, common.SETTINGS))